# PathSnap — Julia's HMM and Viterbi run notebook

This notebook overlays Julia's HMM/Viterbi code and MLP inference artifact onto the original PathSnap repository ZIP, then runs decoder tuning on TRAIN trips and evaluates the inference pipeline on all 10 `student_test` trips.

**Upload two ZIP files in the next cell:**
1. The original `pathsnap(1).zip` repository snapshot (contains your dataset).
2. `Julia_PathSnap_HMM_Viterbi_Addon.zip` (the generated add-on package).

The raw `data/` folder stays local to Colab and must not be committed to GitHub.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
addon_names = [n for n in zip_names if 'julia_pathsnap_hmm_viterbi_addon' in n.lower()]
repo_names = [n for n in zip_names if 'pathsnap' in n.lower() and n not in addon_names]
assert addon_names, 'Upload Julia_PathSnap_HMM_Viterbi_Addon.zip as well.'
assert repo_names, 'Upload the original pathsnap(1).zip repository snapshot as well.'

workspace = Path('/content/pathsnap_julia_run')
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True)
with zipfile.ZipFile(repo_names[0]) as z:
    z.extractall(workspace / 'repo_extract')
repo_candidates = [p for p in (workspace / 'repo_extract').iterdir() if p.is_dir() and (p/'src').is_dir()]
assert repo_candidates, 'Could not find the repository root containing src/.'
project = repo_candidates[0]
with zipfile.ZipFile(addon_names[0]) as z:
    z.extractall(workspace / 'addon_extract')
addon_roots = [p for p in (workspace / 'addon_extract').iterdir() if p.is_dir() and (p/'src').is_dir()]
addon = addon_roots[0] if addon_roots else workspace / 'addon_extract'
shutil.copytree(addon/'src', project/'src', dirs_exist_ok=True)
shutil.copytree(addon/'artifacts', project/'artifacts', dirs_exist_ok=True)
shutil.copy2(addon/'run_julia_hmm.py', project/'run_julia_hmm.py')
print('Project root:', project)
print('Train trips:', len(list((project/'data'/'train').glob('*'))))
print('Student-test trips:', len(list((project/'data'/'student_test').glob('*'))))
os.chdir(project)

## 1. Install dependencies

The MLP joblib artifact was created with scikit-learn 1.7.2, so keep the pinned version in `requirements.txt` to avoid model-persistence compatibility warnings.

In [ ]:
%pip install -r requirements.txt

## 2. Run HMM tuning + all 10 student-test trips

This runs the decoder grid on two TRAIN trips (fast smoke-grid by default), saves the selected settings to `artifacts/hmm.json`, then evaluates all 10 student-test trips. The evaluator explicitly removes `trip.route` before calling `predict_trip`; the reference route is used only afterwards by `score_trip`.

In [ ]:
!python run_julia_hmm.py --tune-trips 2

## 3. Inspect the per-trip results

In [ ]:
import pandas as pd, json
from IPython.display import display, Image
results = pd.read_csv('results/hmm/classifier_vs_hmm.csv')
display(results[['trip_id','top1_classifier','top1_hmm','route_continuity_clf','route_continuity_hmm','route_f1','inference_ms']])
print('Trips evaluated:', len(results))
print('Mean classifier Top-1:', results.top1_classifier.mean())
print('Mean HMM Top-1:', results.top1_hmm.mean())
print('Mean classifier continuity:', results.route_continuity_clf.mean())
print('Mean HMM continuity:', results.route_continuity_hmm.mean())
print('Integration failures:', len(pd.read_csv('results/hmm/integration_failures.csv')))
display(Image('results/hmm/classifier_vs_hmm.png'))

## 4. Run HMM unit tests

In [ ]:
!python -m pytest tests/test_hmm.py -q

## 5. Download the generated results

The outputs are in `results/hmm/`, and the example non-improvement maps are in `maps/results/`. Download the CSVs, plots and maps you need for the report. Do not upload the raw `data/` directory to GitHub.